# Public-release note

This curated notebook contains the original research workflow with execution outputs removed. Bloomberg and other licensed source data are not distributed. To run it, supply compatible files under `data/processed/` as described in `data/README.md`. Any results produced locally depend on the user's licensed data and are not included in this repository.


# Rolling-Window Rebalancing Analysis

This notebook evaluates portfolio rebalancing strategies over repeated historical windows, following the rolling-window design used in the replication study.

In [ ]:
from pathlib import Path
import sys
import pandas as pd

# Locate the repository root when running from either the project or notebooks directory
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))

from src.portfolio.rolling_window import rolling_rebalancing_results

PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
REPLICATION_DIR = PROJECT_ROOT / "results" / "replication"
REPLICATION_DIR.mkdir(parents=True, exist_ok=True)

SAMPLE_START = "1982-01-31"
SAMPLE_END = "2011-12-31"

In [ ]:
# Load country datasets and retain the paper's original sample
datasets = {
    "US": pd.read_csv(PROCESSED_DIR / "us_data.csv", parse_dates=["Date"]),
    "UK": pd.read_csv(PROCESSED_DIR / "uk_data.csv", parse_dates=["Date"]),
    "DE": pd.read_csv(PROCESSED_DIR / "de_data.csv", parse_dates=["Date"]),
}

sample = {
    country: df[df["Date"].between(SAMPLE_START, SAMPLE_END)].reset_index(drop=True)
    for country, df in datasets.items()
}

pd.DataFrame(
    [{"Country": c, "Start": df["Date"].min(), "End": df["Date"].max(), "Rows": len(df)} for c, df in sample.items()]
)

In [ ]:
# Table 4 compares buy-and-hold with monthly, quarterly, and yearly 60/40 rebalancing
strategies = {
    "Buy-and-hold": {"strategy": "buy_and_hold"},
    "Yearly rebalancing": {"strategy": "periodic", "frequency": "Y"},
    "Quarterly rebalancing": {"strategy": "periodic", "frequency": "Q"},
    "Monthly rebalancing": {"strategy": "periodic", "frequency": "M"},
}

In [ ]:
# Apply strategies to annual rolling windows of five and ten years
results = pd.concat(
    [
        rolling_rebalancing_results(df, horizon, strategies, bond_col=f"Bond_{horizon}Y_Return").assign(Country=country)
        for country, df in sample.items()
        for horizon in [5, 10]
    ],
    ignore_index=True,
)

results.head()

In [ ]:
# Average metrics across rolling windows, following Table 4
summary = (
    results.groupby(["Horizon", "Strategy", "Country"], as_index=False)
    [["Ann_Return", "Ann_Volatility", "Sharpe", "Turnover", "Transaction_Cost"]]
    .mean()
)

strategy_order = ["Buy-and-hold", "Yearly rebalancing", "Quarterly rebalancing", "Monthly rebalancing"]
summary["Strategy"] = pd.Categorical(summary["Strategy"], strategy_order, ordered=True)
summary = summary.sort_values(["Horizon", "Strategy", "Country"]).reset_index(drop=True)
summary

In [ ]:
def panel(metric: str, scale: float = 1) -> pd.DataFrame:
    out = summary.pivot(index=["Horizon", "Strategy"], columns="Country", values=metric)
    return (out[["US", "UK", "DE"]] * scale).round(4)

table4_return = panel("Ann_Return", 100)
table4_volatility = panel("Ann_Volatility", 100)
table4_sharpe = panel("Sharpe")

table4_return

In [ ]:
table4_volatility

In [ ]:
table4_sharpe

In [ ]:
# Save window-level results and summary tables
results.to_csv(REPLICATION_DIR / "rolling_window_results.csv", index=False)
summary.to_csv(REPLICATION_DIR / "rolling_window_summary.csv", index=False)
table4_return.to_csv(REPLICATION_DIR / "rolling_window_table4_return.csv")
table4_volatility.to_csv(REPLICATION_DIR / "rolling_window_table4_volatility.csv")
table4_sharpe.to_csv(REPLICATION_DIR / "rolling_window_table4_sharpe.csv")

with pd.ExcelWriter(REPLICATION_DIR / "rolling_window_table4.xlsx", engine="openpyxl") as writer:
    table4_return.to_excel(writer, sheet_name="mean_return")
    table4_volatility.to_excel(writer, sheet_name="volatility")
    table4_sharpe.to_excel(writer, sheet_name="sharpe")
    summary.to_excel(writer, sheet_name="summary", index=False)
    results.to_excel(writer, sheet_name="window_results", index=False)

REPLICATION_DIR / "rolling_window_table4.xlsx"